In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
from federated_multihead_model import SharedEncoders, Head
from config import D_TABULAR, D_EMBEDDING, D_FUSION

In [3]:
import os, random, numpy as np, pandas as pd
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader

In [4]:
# ==== Config ====
CSV_PATH  = "tabular_dataset/diabetes_012_ready_to_model.csv"  # CSV
LABEL_COL = "Diabetes_012"                                     # Label column name

VAL_RATIO = 0.2
EPOCHS    = 5
BATCH     = 64
LR        = 3e-4
WD        = 1e-4
LOG_EVERY = 50

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False
set_seed(42)

# === from config.py ===
from config import D_EMBEDDING, D_FUSION, D_TABULAR
try:
    import tabular_info
    N_CLASSES = tabular_info.n_tabular_classes
except:
    N_CLASSES = 2


In [5]:
#Dataset
class TabularOnlyDataset(Dataset):
    """
    Only table features + labels are read.
    """
    def __init__(self, csv_path, label_col):
        df = pd.read_csv(csv_path)
        assert label_col in df.columns
        self.label_col = label_col

        drop_cols = {label_col}
        feat_cols = [c for c in df.columns if (c not in drop_cols) and pd.api.types.is_numeric_dtype(df[c])]
        
        self.X = df[feat_cols].astype(np.float32).values
        self.y = df[label_col].astype(int).values.astype(np.int64)

    def __len__(self): return len(self.y)

    def __getitem__(self, i):
        ehr   = torch.from_numpy(self.X[i])
        label = torch.tensor(self.y[i], dtype = torch.long)
        return {"ehr": ehr, "label": label}

In [6]:
# DataLoader
full = TabularOnlyDataset(CSV_PATH, LABEL_COL)

N = len(full)
idx = np.arange(N)
np.random.shuffle(idx)

cut = int(N*(1.0-VAL_RATIO))
tr_idx, va_idx = idx[:cut], idx[cut:]

train_ds = torch.utils.data.Subset(full, tr_idx)
val_ds   = torch.utils.data.Subset(full, va_idx)

train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True )
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False)

d_text = next(iter(train_loader))["ehr"].shape[1]  # Table feature dimensions

In [ ]:
# reload file if changed
import importlib, multihead_model
importlib.reload(multihead_model)

In [8]:
# new structure
class TabularClientModel(nn.Module):
    def __init__(self, shared_encoders: SharedEncoders, n_classes: int):
        super().__init__()
        self.enc = shared_encoders
        self.head = Head(d_embedding = D_EMBEDDING, 
                         n_classes = tabular_info.n_tabular_classes)

    def forward(self, x_text):
        z = self.enc.forward_tabular(x_text)
        return self.head(z)

In [9]:
# global_encoders is an instance of SharedEncoders
global_encoders = SharedEncoders(
    d_tabular = D_TABULAR, 
    d_embedding = D_EMBEDDING, 
    d_fusion = D_FUSION
    )

global_state = global_encoders.state_dict()
# only here in this demo, should not exist in practice

In [ ]:
# At the client:
# 1. load global encoders
local_encoders = SharedEncoders(
    d_tabular   = D_TABULAR, 
    d_embedding = D_EMBEDDING, 
    d_fusion    = D_FUSION)
# Load the weights
local_encoders.load_state_dict(global_state, strict=False)

# 2. Construct the client model (with 2-class tabular head)
client_model = TabularClientModel(
    shared_encoders = local_encoders, 
    n_classes = tabular_info.n_tabular_classes
    )

In [14]:
'''
model = MultiHeadModel(
    d_tabular   = D_TABULAR,
    d_embedding = D_EMBEDDING,
    d_fusion    = D_FUSION,
    n_tabular_classes = N_CLASSES,
    n_image_classes   = None,
    n_multi_classes   = None
).to(device)
'''

model = client_model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)

In [15]:
def accuracy_from_logits(logits, labels):
    return (logits.argmax(1) == labels).float().mean().item()

@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    tot_l=tot_a=n=0
    for batch in loader:
        for k,v in batch.items():
            if isinstance(v, torch.Tensor): batch[k]=v.to(device)

        logits = model(batch["ehr"])
        loss   = F.cross_entropy(logits, batch["label"])
        
        acc    = accuracy_from_logits(logits, batch["label"])
        tot_l += loss.item(); tot_a += acc; n += 1
    return tot_l/max(1,n), tot_a/max(1,n)

def train_one_epoch(model, loader, opt):
    model.train()
    run_l=run_a=n=0
    for i,batch in enumerate(loader,1):
        for k,v in batch.items():
            if isinstance(v, torch.Tensor): batch[k]=v.to(device)

        opt.zero_grad(set_to_none=True)

        logits = model(batch["ehr"])
        loss   = F.cross_entropy(logits, batch["label"])
        
        loss.backward()
        opt.step()

        acc = accuracy_from_logits(logits.detach(), batch["label"])
        run_l += loss.item(); run_a += acc; n += 1
        if i % LOG_EVERY == 0:
            print(f"step {i:>5d}: loss={run_l/n:.4f} acc={run_a/n:.4f}")
    return run_l/max(1,n), run_a/max(1,n)


In [16]:
os.makedirs("runs/exp_final", exist_ok=True)
best=-1.0
best_path="runs/exp_final/best.pth"

for epoch in range(1, EPOCHS+1):
    print(f"\nEpoch {epoch}")
    tr_l,tr_a = train_one_epoch(model, train_loader, optimizer)
    va_l,va_a = evaluate(model, val_loader)
    print(f" -> train loss={tr_l:.4f} acc={tr_a:.4f}")
    print(f" ->   val loss={va_l:.4f} acc={va_a:.4f}")
    if va_a > best:
        best = va_a
        torch.save({"epoch":epoch,"model":model.state_dict()}, best_path)
        print(f" [best updated] {best:.4f} -> {best_path}")



Epoch 1
step    50: loss=0.4869 acc=0.8113
step   100: loss=0.4434 acc=0.8178
step   150: loss=0.4227 acc=0.8249
step   200: loss=0.4154 acc=0.8260
step   250: loss=0.4074 acc=0.8266
step   300: loss=0.4020 acc=0.8272
step   350: loss=0.3971 acc=0.8282
step   400: loss=0.3950 acc=0.8286
step   450: loss=0.3898 acc=0.8300
step   500: loss=0.3869 acc=0.8314
step   550: loss=0.3857 acc=0.8323
step   600: loss=0.3842 acc=0.8331
step   650: loss=0.3813 acc=0.8340
step   700: loss=0.3817 acc=0.8335
step   750: loss=0.3817 acc=0.8333
step   800: loss=0.3796 acc=0.8339
step   850: loss=0.3797 acc=0.8333
step   900: loss=0.3795 acc=0.8330
step   950: loss=0.3789 acc=0.8333
step  1000: loss=0.3790 acc=0.8330
step  1050: loss=0.3790 acc=0.8328
step  1100: loss=0.3788 acc=0.8328
step  1150: loss=0.3783 acc=0.8332
step  1200: loss=0.3781 acc=0.8332
step  1250: loss=0.3778 acc=0.8336
step  1300: loss=0.3776 acc=0.8335
step  1350: loss=0.3770 acc=0.8336
step  1400: loss=0.3765 acc=0.8336
step  1450: